In [0]:
from pyspark.sql import functions as F

# 1. Configuration (Set to your exact Unity Catalog name)
catalog = 'olist'  
base = f'/Volumes/{catalog}/bronze/landing'

# List of target tables matching your raw CSV folders
tables = [
    'orders', 'order_items', 'payments', 'reviews', 
    'customers', 'products', 'sellers', 'category_translation'
]

# 2. Reusable Ingestion Function using Auto Loader (cloudFiles)
def ingest(t):
    print(f"Starting Auto Loader ingestion for table: {t}...")
    
    (spark.readStream.format('cloudFiles')
      .option('cloudFiles.format', 'csv')
      .option('header', 'true')
      .option('cloudFiles.inferColumnTypes', 'true') # Infers proper data types automatically
      .option('cloudFiles.schemaLocation', f'{base}/metadata/_schema/{t}')
      .load(f'{base}/{t}/')
      # Append system metadata columns for lineage and tracking
      .withColumn('_ingested_at', F.current_timestamp())
      .withColumn('_source_file', F.col('_metadata.file_path'))
      .writeStream
      .format('delta') # Explicitly state Delta format
      .option('checkpointLocation', f'{base}/metadata/_chk/{t}')
      .trigger(availableNow=True) # Processes all available data as a batch, then stops
      .toTable(f'{catalog}.bronze.{t}'))
      
    print(f"Successfully ingested and merged into Delta table: {catalog}.bronze.{t}\n")

# 3. Execution Loop
for t in tables:
    ingest(t)
